# Productor de eventos CoinGecko

Este cuaderno arranca un broker Kafka local con Docker, crea el tópico `crypto_prices` y envía cada evento JSON directamente a Kafka.

Ejecuta este cuaderno primero. Después lanza el cuaderno consumidor para leer los eventos desde Kafka con Spark Structured Streaming.


## 1. Instalación de dependencias

El productor usa `requests` para consultar CoinGecko y `kafka-python` para publicar mensajes en Kafka.

In [ ]:
pip install requests kafka-python


## 2. Arranque local de Kafka con Docker

Esta celda deja disponible un broker Kafka en `localhost:9092`. Si el contenedor ya existe, lo reutiliza.

In [ ]:
import shutil
import socket
import subprocess
import time

KAFKA_CONTAINER_NAME = "kafka-m3t1"
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "crypto_prices"

def port_is_open(host="localhost", port=9092, timeout=1):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0

def run_command(command, check=True):
    result = subprocess.run(command, text=True, capture_output=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result

def ensure_docker_daemon_access():
    if shutil.which("docker") is None:
        raise RuntimeError("Docker no está instalado o no está en el PATH. Instala Docker o arranca Kafka manualmente en localhost:9092.")
    result = run_command(["docker", "ps"], check=False)
    if result.returncode != 0:
        raise RuntimeError(
            "Docker está instalado, pero este usuario no puede acceder al daemon. "
            "En Linux, añade tu usuario al grupo docker y vuelve a iniciar sesión, "
            "o arranca Kafka manualmente antes de ejecutar este notebook. "
            "Detalle: " + (result.stderr.strip() or result.stdout.strip())
        )

def docker_container_exists(name):
    result = run_command(["docker", "ps", "-a", "--filter", f"name={name}", "--format", "{{.Names}}"], check=False)
    return name in result.stdout.splitlines()

if port_is_open():
    print(f"Kafka ya responde en {KAFKA_BOOTSTRAP_SERVERS}.")
else:
    ensure_docker_daemon_access()

    if docker_container_exists(KAFKA_CONTAINER_NAME):
        print(f"Arrancando contenedor existente: {KAFKA_CONTAINER_NAME}")
        run_command(["docker", "start", KAFKA_CONTAINER_NAME])
    else:
        print(f"Creando contenedor Kafka: {KAFKA_CONTAINER_NAME}")
        run_command([
            "docker", "run", "-d", "--name", KAFKA_CONTAINER_NAME,
            "-p", "9092:9092",
            "-e", "KAFKA_NODE_ID=1",
            "-e", "KAFKA_PROCESS_ROLES=broker,controller",
            "-e", "KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093",
            "-e", "KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://localhost:9092",
            "-e", "KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER",
            "-e", "KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT",
            "-e", "KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093",
            "-e", "KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1",
            "-e", "KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=1",
            "-e", "KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=1",
            "-e", "KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS=0",
            "apache/kafka:3.7.0",
        ])

    for attempt in range(30):
        if port_is_open():
            print(f"Kafka disponible en {KAFKA_BOOTSTRAP_SERVERS}.")
            break
        time.sleep(2)
    else:
        logs = run_command(["docker", "logs", "--tail", "80", KAFKA_CONTAINER_NAME], check=False)
        raise RuntimeError("Kafka no quedó disponible en localhost:9092. Últimos logs:\n" + logs.stdout + logs.stderr)


## 3. Configuración del productor

El tópico debe coincidir con el que lee el cuaderno consumidor.

In [ ]:
import json
from datetime import datetime, timezone

import requests
from kafka import KafkaAdminClient, KafkaProducer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

COINS = ["bitcoin", "ethereum", "solana", "cardano"]
CURRENCY = "eur"
SLEEP_SECONDS = 15
MAX_ITERATIONS = 20  # Cambia a None para ejecución indefinida

print("Bootstrap servers:", KAFKA_BOOTSTRAP_SERVERS)
print("Topic:", KAFKA_TOPIC)


## 4. Preparación del tópico y del productor Kafka

Cada mensaje se envía como JSON en `value`; la criptomoneda se usa como `key`.

In [ ]:
admin_client = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS)

try:
    admin_client.create_topics([
        NewTopic(name=KAFKA_TOPIC, num_partitions=1, replication_factor=1)
    ])
    print(f"Topic creado: {KAFKA_TOPIC}")
except TopicAlreadyExistsError:
    print(f"Topic ya existente: {KAFKA_TOPIC}")
finally:
    admin_client.close()

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    key_serializer=lambda key: key.encode("utf-8"),
    value_serializer=lambda value: json.dumps(value).encode("utf-8"),
    acks="all",
    retries=3,
)

print("Productor Kafka preparado.")


## 5. Envío de eventos a Kafka

Deja esta celda ejecutándose mientras abres el cuaderno consumidor.

In [ ]:
url = "https://api.coingecko.com/api/v3/simple/price"
params = {"ids": ",".join(COINS), "vs_currencies": CURRENCY}

try:
    for i in range(MAX_ITERATIONS if MAX_ITERATIONS is not None else 10**9):
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            event_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
            sent_events = 0

            for coin, values in data.items():
                event = {
                    "event_time": event_time,
                    "coin": coin,
                    "currency": CURRENCY,
                    "price": float(values[CURRENCY]),
                }
                producer.send(KAFKA_TOPIC, key=coin, value=event)
                sent_events += 1

            producer.flush()
            print(f"[{i + 1}] Eventos enviados a Kafka: {sent_events}")

        except Exception as exc:
            print("Error consultando CoinGecko o enviando a Kafka:", exc)

        time.sleep(SLEEP_SECONDS)
finally:
    producer.flush()
    producer.close()
    print("Productor Kafka cerrado.")
